# Week 2 Assignment: Mini Semantic Search Engine

**AI/ML AWS AASTU Cohort · Deliverable**

> *From messy FAQ data to an intelligent search engine that understands student intent.*

---

### Objective
In this hands-on lab, we build a **Mini Semantic Search Engine** over an Addis Ababa Science and Technology University (AASTU) student knowledge base. Unlike traditional keyword matching (which fails when users phrase questions differently), semantic search maps text into high-dimensional vector space so that questions with **similar meaning** land close to each other.

### Deliverable Pipeline
1. **Data Ingestion**: Load a real-world FAQ dataset (`JSON` / `CSV`) using **Pandas**.
2. **Data Cleaning**: Deduplicate, trim whitespace, impute nulls, and clean messy strings.
3. **Text Chunking**: Segment documents with a **sliding window and overlap** so context is never cut in half.
4. **Vector Storage**: Ingest documents and metadata into **ChromaDB**.
5. **Semantic Querying**: Query the database using natural language and retrieve the **Top-3 nearest neighbors** by Cosine Similarity.
6. **Hybrid Filtering**: Filter results using both unstructured text similarity and structured metadata (e.g. department/category).

## 1. Setup & Environment
Let us import our core libraries: `pandas`, `numpy`, `re`, and vector database tools.

In [ ]:
# Install dependencies (especially for Google Colab)
!pip install chromadb pandas numpy -q

import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import chromadb

print(f"ChromaDB version: {chromadb.__version__}")
print(f"Pandas version:   {pd.__version__}")
print(f"NumPy version:    {np.__version__}")


## 2. Load & Explore the FAQ Dataset
We load our dataset using `pandas.read_json()` (or `pandas.read_csv()`) and perform our 5 standard exploratory checks:
- `.shape`: Dimensions of the dataset
- `.head()`: Sample rows
- `.info()`: Column data types & non-null counts
- `.isna().sum()`: Check for missing data

In [ ]:
import os
import urllib.request
import pandas as pd

# Check local paths first
candidates = ['data/faqs.json', 'week-02/data/faqs.json', 'faqs.json']
dataset_path = next((p for p in candidates if os.path.exists(p)), None)

# If running on Google Colab, download directly from GitHub repository
if dataset_path is None:
    print('Dataset not found locally; downloading from GitHub repository...')
    url = 'https://raw.githubusercontent.com/AbellTT/aws-sbg-tasks/main/week-02/data/faqs.json'
    urllib.request.urlretrieve(url, 'faqs.json')
    dataset_path = 'faqs.json'

df = pd.read_json(dataset_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n')
print('Missing values per column:')
print(df.isna().sum())
print('\nFirst 3 FAQ entries:')
df[['id', 'category', 'department', 'question']].head(3)


## 3. Data Cleaning Pipeline
In production AI systems, dirty data leads to poor search retrieval. We implement:
1. **Deduplication**: Remove duplicate questions
2. **Text Normalization**: Strip leading/trailing whitespaces and collapse multiple spaces
3. **Missing Value Imputation**: Fill any blank fields with standard defaults
4. **Unified Text Column**: Create a combined representation of question + answer for comprehensive semantic indexing.

In [ ]:
def clean_faq_dataframe(data: pd.DataFrame) -> pd.DataFrame:
    cleaned = data.copy()
    cleaned = cleaned.drop_duplicates(subset=["question"]).reset_index(drop=True)
    for col in ["question", "answer", "category", "department"]:
        if col in cleaned.columns:
            cleaned[col] = cleaned[col].astype(str).str.strip()
            cleaned[col] = cleaned[col].replace(["nan", "None", ""], "Unspecified")
    cleaned["full_text"] = cleaned["question"] + " " + cleaned["answer"]
    return cleaned

clean_df = clean_faq_dataframe(df)
print(f"Original rows: {len(df)} | Clean rows: {len(clean_df)}")
clean_df[["id", "category", "full_text"]].head(2)

## 4. Text Preparation: Chunking with Overlap
Why not embed an entire 50-page document as a single vector?
- **Too large**: The specific answer drowns in irrelevant background noise.
- **Too small**: Context is lost if words are severed.
- **Sliding Window with Overlap**: Slicing with overlapping words guarantees that sentences spanning boundary edges remain connected.

In [ ]:
def chunk_text(text: str, size: int = 80, overlap: int = 20) -> list:
    words = text.split()
    if len(words) <= size:
        return [text]
    step = max(1, size - overlap)
    chunks = []
    for i in range(0, len(words), step):
        chunk_slice = words[i:i + size]
        if chunk_slice:
            chunks.append(" ".join(chunk_slice))
        if i + size >= len(words):
            break
    return chunks

prepared_docs = []
for idx, row in clean_df.iterrows():
    chunks = chunk_text(row["full_text"], size=75, overlap=20)
    for c_idx, chunk in enumerate(chunks):
        prepared_docs.append({
            "id": f"{row['id']}_c{c_idx}",
            "text": chunk,
            "question": row["question"],
            "answer": row["answer"],
            "category": row["category"],
            "department": row["department"]
        })

print(f"Total input FAQs: {len(clean_df)}")
print(f"Total indexed chunks: {len(prepared_docs)}")
print(f"Sample chunk: ID={prepared_docs[0]['id']}, Text={prepared_docs[0]['text'][:80]}...")

## 5. Embed & Store in ChromaDB
As instructed in the lecture deliverable, we use **ChromaDB** to embed and store our document chunks:

1. **Initialize ChromaDB**: Create an in-memory client (`chromadb.Client()`).
2. **Collection with Cosine Distance**: Configure `hnsw:space: cosine` for direction-based similarity.
3. **Automatic Vector Embeddings**: Chroma automatically converts each text chunk into a 384-dimensional dense vector using its default embedding model (`all-MiniLM-L6-v2`).
4. **Metadata Storage**: Store structured attributes (question, answer, category, department) alongside each vector for hybrid search.

In [ ]:
# Initialize ChromaDB client
client = chromadb.Client()

# Create or get collection configured with Cosine Similarity
collection = client.get_or_create_collection(
    name="aastu_faq_collection",
    metadata={"hnsw:space": "cosine"}
)

# Extract chunk texts, metadatas, and IDs
chunk_texts = [d["text"] for d in prepared_docs]
chunk_ids = [d["id"] for d in prepared_docs]
chunk_metadatas = [
    {
        "question": d["question"][:500],
        "answer": d["answer"][:500],
        "category": d["category"],
        "department": d["department"],
    }
    for d in prepared_docs
]

# Embed and store documents in ChromaDB
collection.add(
    documents=chunk_texts,
    metadatas=chunk_metadatas,
    ids=chunk_ids
)

print(f"Successfully embedded and stored {collection.count()} chunks in ChromaDB collection '{collection.name}'!")


## 6. Querying ChromaDB (Top-3 Nearest Neighbors)
We query ChromaDB using natural conversational questions. Chroma converts the query text to an embedding on the fly and retrieves the **top 3 closest document vectors** using HNSW approximate nearest neighbors.

In [ ]:
def query_faqs(query_text: str, top_k: int = 3, category: str = None):
    """Query ChromaDB and display top matches with relevance distance."""
    where_filter = {"category": category} if category else None
    
    results = collection.query(
        query_texts=[query_text],
        n_results=top_k,
        where=where_filter
    )
    
    print("=" * 75)
    print(f"QUERY: \"{query_text}\"")
    if category:
        print(f"FILTER: [Category = {category}]")
    print("=" * 75)
    
    docs = results["documents"][0]
    metas = results["metadatas"][0]
    distances = results["distances"][0] if "distances" in results else [0.0] * len(docs)
    
    for i, (doc, meta, dist) in enumerate(zip(docs, metas, distances), start=1):
        similarity = round(max(0.0, 1.0 - dist), 4)
        print(f"\n[Match {i}] Cosine Similarity: {similarity} (Distance: {round(dist, 4)})")
        print(f"  Category:   {meta['category']} | Department: {meta['department']}")
        print(f"  Question:   {meta['question']}")
        print(f"  Answer:     {meta['answer']}")
    print("\n")

# Test Query 1: Wi-Fi connectivity
query_faqs("How do I connect to campus wifi on my mobile phone?")

# Test Query 2: Low GPA & probation
query_faqs("What happens if my CGPA is below 2.0?")

# Test Query 3: Off-campus research paper access
query_faqs("Can I download IEEE and ScienceDirect research papers from home?")


## 7. Hybrid Search: Semantic Similarity + Structured Metadata Filtering
ChromaDB allows combining dense vector search with SQL-like metadata filters (`where={"category": ...}`).
Here, we search for *"Where do I register and borrow materials?"* restricted strictly to the **`Campus Services`** category:

In [ ]:
query_faqs("Where do I register and borrow materials?", top_k=3, category="Campus Services")


## 8. Summary & Key Takeaways

1. **Structured + Unstructured**: Real-world AI applications rely on both lanes — structured metadata (category, date, author) and unstructured text.
2. **Text Chunking with Overlap**: Splitting long documents into manageable windows avoids drowning the answer while retaining context.
3. **Vector Embeddings**: Project textual meaning into vector space where cosine angle directly models semantic similarity.
4. **Vector Databases (Chroma / HNSW)**: Eliminate slow brute-force $O(N)$ comparisons using approximate nearest neighbors indexing for lightning-fast sub-millisecond retrieval.